In [1]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import minimize

phi_vals = np.linspace(1.5, 2.0, 500)
T_vals = np.linspace(0.01, 0.5, 10)
gamma = 0.5772156649  # Constante de Euler-Mascheroni
cutoff = 1e-10        # Para evitar log(0) o exp(0)

def I(k, p, lamda, m, T):
    arg = k**2 + lamda * p**2 - m**2
    if arg <= 0:
        return 0.0
    raiz = np.sqrt(arg)
    exp_arg = np.exp(-raiz / T)
    if exp_arg >= 1.0 - cutoff:
        return 0.0
    return (T / (2 * np.pi**2)) * k**2 * np.log(1 - exp_arg)

def V_eff_num(phi_vals, lamda, m, T):
    datos = []
    for p in phi_vals:
        integrando = lambda k: I(k, p, lamda, m, T)
        integral, _ = quad(integrando, 0, 200, limit=200)
        datos.append(integral)
    datos = np.array(datos)

    M2 = lamda * phi_vals**2 - m**2
    M = np.sqrt(M2.astype(np.complex128))
    M0 = np.sqrt(np.complex128(-m**2))
    integral_0, _ = quad(lambda k: I(k, 0, lamda, m, T), 0, 200, limit=200)

    V0 = -m**2*0/2 + lamda*0/4 - (M0**2)/(64*np.pi**2)*(3/2 + np.log(1/M0)) + integral_0
    V = -m**2*phi_vals**2/2 + lamda*phi_vals**4/4 - (M2/(64*np.pi**2))*(3/2 + np.log(1/M)) + datos - V0
    return np.real(V)

def V_eff_analitico(phi_vals, lamda, m, T):
    M2 = lamda * phi_vals**2 - m**2
    M = np.sqrt(M2.astype(np.complex128))
    M0 = np.sqrt(np.complex128(-m**2))

    log_M2 = np.log(np.complex128(M2 / (16 * np.pi**2 * T**2)))
    log_M0 = np.log(M0 / (16 * np.pi**2 * T**2))

    V0 = -m**2*0/2 + lamda*0/4 - (M0**2)/(64*np.pi**2)*(3/2 + np.log(1/M0)) \
         - (T**4*np.pi**2)/90 + (M0*T**2)/24 - (T*M0**(3/2))/(12*np.pi) \
         - (M0**2/(64*np.pi**2))*(log_M0 - 2*gamma - 3/2)

    V = -m**2*phi_vals**2/2 + lamda*phi_vals**4/4 - (M2/(64*np.pi**2))*(3/2 + np.log(1/M)) \
        - (T**4*np.pi**2)/90 + (M*T**2)/24 - (T*M**(3/2))/(12*np.pi) \
        - (M2/(64*np.pi**2))*(log_M2 - 2*gamma - 3/2) - V0

    return np.real(V)


In [3]:
def loss(params):
    m, lamda = params
    total_error = 0.0
    for T in T_vals:
        Vn = V_eff_num(phi_vals, lamda, m, T)
        Va = V_eff_analitico(phi_vals, lamda, m, T)
        total_error += np.mean((Vn - Va)**2)
    return total_error

# Llamada de ejemplo
res = minimize(loss, x0=[1.0, 0.5], method='Nelder-Mead', options={'disp': True})
print("Parámetros óptimos encontrados (m, lambda):", res.x)


C:\Users\limen\AppData\Local\Temp\ipykernel_6160\3001008356.py:24: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  integral, _ = quad(integrando, 0, 200, limit=200)
C:\Users\limen\AppData\Local\Temp\ipykernel_6160\3001008356.py:24: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  integral, _ = quad(integrando, 0, 200, limit=200)
C:\Users\limen\AppData\Local\Temp\ipykernel_6160\3001008356.py:31: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  integral_0, _ = quad(lambda k: I(k, 0, lamda, m, T), 0, 200, limit=200)


Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 43
         Function evaluations: 78
Parámetros óptimos encontrados (m, lambda): [9.99954418e-01 1.62599340e-05]
